# Understanding UrbanEV

### Objective

Explore the structure and contents of the UrbanEV dataset,
an open benchmark of electric vehicle charging activity
in Shenzhen, China.

**Source:** Li et al. (2025), Scientific Data

**Repository:** https://github.com/IntelligentSystemsLab/UrbanEV


### Dataset Overview

UrbanEV is an open dataset of electric vehicle charging
activity in Shenzhen, China, published by Li et al. (2025)
in Scientific Data.

#### Temporal coverage

- Period: September 1, 2022 – February 28, 2023
- Duration: 181 days
- Resolution: Hourly
- Observations: 4,344
- Missing timestamps: 0
- Duplicate timestamps: 0

#### Spatial coverage

The cleaned dataset contains:

- 275 Traffic Analysis Zones (TAZs)
- 1,362 charging stations
- 17,532 charging piles

Charging infrastructure is unevenly distributed across
the selected zones. The median zone has 4 charging stations
and 48 charging piles.

#### Available data

The dataset combines:

1. Charging activity: volume, duration and occupancy
2. Charging prices: electricity and service fees
3. Weather: measurements from two meteorological stations
4. Spatial information: zone adjacency and distances
5. Urban context: 712,135 Points of Interest

#### Dataset structure

The six charging-related datasets share the same 4,344
timestamps and 275 zone identifiers.

Both weather datasets share the charging timestamps.

The spatial matrices are 275 × 275, representing
relationships between geographic zones.

#### Important considerations

- Charging volume is estimated rather than necessarily
  directly metered.
- The dataset has undergone preprocessing and filtering.
- Complete timestamps do not guarantee complete measurements.
- POIs are not directly assigned to geographic zones.
- Geographic coordinate systems differ between data sources.
- Occupancy units should be verified before analysis.



### File inventory

In [2]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

files = sorted(DATA_DIR.glob("*.csv"))

inventory = pd.DataFrame({
    "filename": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1024**2, 2) for f in files]
})

display(inventory)


,filename,size_mb
0,adj.csv,0.15
1,distance.csv,0.86
2,duration.csv,16.98
3,e_price.csv,11.49
4,inf.csv,0.07
5,inf_raw.csv,0.08
6,occupancy.csv,5.26
7,poi.csv,31.13
8,s_price.csv,14.27
9,volume-11kW.csv,18.81


### The structure of `volume.csv`

`volume.csv` contains estimated hourly EV charging electricity consumption in kWh.


In [3]:
volume = pd.read_csv(DATA_DIR / "volume.csv")

print("Shape (rows, columns):", volume.shape)

print("\nFirst 10 column names:")
print(volume.columns[:10].tolist())

print("\nFirst 5 rows (first 6 columns):")
display(volume.iloc[:5, :6])

print("\nData types (first 6 columns):")
display(volume.dtypes.head(6))

Shape (rows, columns): (4344, 276)

First 10 column names:
['time', '102', '104', '105', '106', '107', '108', '109', '110', '111']

First 5 rows (first 6 columns):


,time,102,104,105,106,107
0,2022-09-01 00:00:00,56.291667,11.083333,60.375000,78.750000,90.125000
1,2022-09-01 01:00:00,50.458333,9.333333,72.625000,81.666667,100.333333
2,2022-09-01 02:00:00,55.708333,12.833333,73.208333,89.250000,94.500000
3,2022-09-01 03:00:00,57.750000,12.541667,67.666667,74.083333,89.541667
4,2022-09-01 04:00:00,58.333333,11.958333,69.708333,78.166667,88.666667



Data types (first 6 columns):


time        str
102     float64
104     float64
105     float64
106     float64
107     float64
dtype: object

**How to interpret a row**

Each value represents the estimated charging electricity volume in that zone during that hour.

These volumes are estimated using charging activity and rated charging power, rather than necessarily representing directly metered electricity consumption.

### Verify temporal coverage

We can confirm that our time series is complete and correctly ordered.

In [4]:

# Convert the timestamp column to datetime
volume["time"] = pd.to_datetime(volume["time"])

# Sort timestamps
timestamps = volume["time"].sort_values()

# Create the expected hourly timeline
expected = pd.date_range(
    start=timestamps.min(),
    end=timestamps.max(),
    freq="h"
)

print("First timestamp:", timestamps.min())
print("Last timestamp:", timestamps.max())
print("Total observations:", len(timestamps))

print("\nDuplicate timestamps:", timestamps.duplicated().sum())
print("Missing hourly timestamps:",
      len(expected.difference(pd.DatetimeIndex(timestamps))))

print("\nTime intervals between observations:")
display(timestamps.diff().value_counts().head())


First timestamp: 2022-09-01 00:00:00
Last timestamp: 2023-02-28 23:00:00
Total observations: 4344

Duplicate timestamps: 0
Missing hourly timestamps: 0

Time intervals between observations:


time
0 days 01:00:00    4343
Name: count, dtype: int64

All 4,343 consecutive time intervals are exactly one hour apart.
This means that we can confidently describe the dataset as containing 181 consecutive days of hourly observations.

A complete timeline doesn't necessarily mean all charging measurements are valid or non-missing. We'll investigate that separately.

### Understanding the spatial dimension

We now know that our dataset contains 275 geographic zones. But what exactly are these zones?
According to the authors, they represent Traffic Analysis Zones (TAZs) in Shenzhen. These are geographic areas used in transportation planning and analysis.

The file `inf.csv` contains information about individual charging stations and their associated zones.
Let's inspect it and see how the station identifiers relate to our `volume.csv` columns.

In [5]:

# Load station information
stations = pd.read_csv(DATA_DIR / "inf.csv")

print("Station information shape:", stations.shape)

print("\nColumns:")
print(stations.columns.tolist())

print("\nFirst 5 rows:")
display(stations.head())

# Identify geographic zones
volume_zones = set(map(int, volume.columns.drop("time")))
station_zones = set(stations["TAZID"].dropna().astype(int))

print("\nSpatial summary:")
print("Zones in volume.csv:", len(volume_zones))
print("Zones in inf.csv:", len(station_zones))
print("Unique stations:", stations["station_id"].nunique())
print("Total charging piles:", stations["charge_count"].sum())

print("\nZones in volume.csv but not in inf.csv:")
print(sorted(volume_zones - station_zones))


Station information shape: (1362, 7)

Columns:
['station_id', 'longitude', 'latitude', 'charge_count', 'TAZID', 'area', 'perimeter']

First 5 rows:


,station_id,longitude,latitude,charge_count,TAZID,area,perimeter
0,1001,113.784724,22.714121,20,559,5726523.056,9624.5755
1,1002,113.785002,22.725900,22,558,3927542.045,8189.3687
2,1003,113.787971,22.735538,6,558,3927542.045,8189.3687
3,1004,113.788126,22.693449,11,596,2972113.241,7335.1002
4,1006,113.790987,22.731351,10,594,2044150.527,5825.1986



Spatial summary:
Zones in volume.csv: 275
Zones in inf.csv: 275
Unique stations: 1362
Total charging piles: 17532

Zones in volume.csv but not in inf.csv:
[]


**What Do the Columns Mean?**

| Column                  | Meaning                                 |
| ----------------------- | --------------------------------------- |
| `station_id`            | Unique charging station identifier      |
| `longitude`, `latitude` | Geographic coordinates of the station   |
| `charge_count`          | Number of charging piles at the station |
| `TAZID`                 | Traffic Analysis Zone identifier        |
| `area`                  | Area of the geographic zone (m²)        |
| `perimeter`             | Perimeter of the geographic zone (m)    |


All 275 zones in `volume.csv` are represented in `inf.csv`. This means we can connect hourly charging demand with information about the charging infrastructure in each zone.

`inf.csv` contains station-level records, whereas `volume.csv` contains zone-level measurements. Multiple stations can belong to the same zone.

### How is charging infrastructure distributed across zones?

We know there are 1,362 charging stations across 275 zones.

But are the stations evenly distributed?

The distribution of infrastructure may help explain differences in charging activity between zones.

In [6]:
# Aggregate station information by geographic zone
zone_summary = (
    stations.groupby("TAZID")
    .agg(
        n_stations=("station_id", "nunique"),
        n_piles=("charge_count", "sum")
    )
    .reset_index()
)

print("Summary statistics:")
display(zone_summary[["n_stations", "n_piles"]].describe().round(2))

print("\nTop 5 zones by number of charging piles:")
display(
    zone_summary.nlargest(5, "n_piles")
)

print("\nBottom 5 zones by number of charging piles:")
display(
    zone_summary.nsmallest(5, "n_piles")
)

Summary statistics:


,n_stations,n_piles
count,275.00,275.00
mean,4.95,63.75
std,4.45,56.73
min,1.00,4.00
25%,2.00,24.00
50%,4.00,48.00
75%,7.00,84.00
max,41.00,373.00



Top 5 zones by number of charging piles:


,TAZID,n_stations,n_piles
271,1167,19,373
207,1011,41,332
67,526,14,262
42,329,17,254
47,335,19,249



Bottom 5 zones by number of charging piles:


,TAZID,n_stations,n_piles
33,316,1,4
94,628,1,6
130,719,1,6
150,792,1,6
163,852,1,8


**Charging infrastructure is distributed quite unevenly across Shenzhen's 275 zones!!**

1. The typical zone has relatively few charging stations. The median zone contains four stations and 48 charging piles. Half of the zones have between 24 and 84 charging piles.

2. Some zones have considerably more infrastructure. Zone 1167 contains 373 charging piles across 19 stations, whereas zone 316 has only four piles at a single station. That's a substantial difference.

3. More stations doesn't necessarily mean more charging piles. Zone 1011 contains 41 stations and 332 piles, while zone 1167 has just 19 stations but 373 piles. Station sizes clearly differ.

### Check the other charging time series

So far, we've explored `volume.csv`. But UrbanEV provides multiple measurements of charging activity.

First we have to establish whether these files share the same structure:

* `volume.csv` — estimated electricity consumption (kWh)
* `volume-11kW.csv` — alternative electricity consumption estimate (kWh)
* `duration.csv` — charging duration (hours)
* `occupancy.csv` — charging occupancy (we'll verify its units)
* `e_price.csv` — electricity price (yuan/kWh)
* `s_price.csv` — service price (yuan/kWh)

In [7]:

charging_files = [
    "volume.csv",
    "volume-11kW.csv",
    "duration.csv",
    "occupancy.csv",
    "e_price.csv",
    "s_price.csv",
]

reference_zones = set(volume.columns) - {"time"}

results = []

for filename in charging_files:
    path = DATA_DIR / filename

    # Read only the header and timestamps
    columns = pd.read_csv(path, nrows=0).columns
    times = pd.read_csv(path, usecols=["time"])["time"]
    times = pd.to_datetime(times)

    zones = set(columns) - {"time"}

    results.append({
        "file": filename,
        "rows": len(times),
        "zones": len(zones),
        "same_zones": zones == reference_zones,
        "same_timestamps": times.equals(volume["time"])
    })

display(pd.DataFrame(results))


,file,rows,zones,same_zones,same_timestamps
0,volume.csv,4344,275,True,True
1,volume-11kW.csv,4344,275,True,True
2,duration.csv,4344,275,True,True
3,occupancy.csv,4344,275,True,True
4,e_price.csv,4344,275,True,True
5,s_price.csv,4344,275,True,True


### Understanding the weather data

Next, we investigate `weather_airport.csv` and `weather_central.csv`.

Unlike the charging data, which has one column per geographic zone, these files contain observations from two meteorological stations in Shenzhen.

Our goal is to understand their structure and whether their timestamps align with the charging data.

In [8]:

weather_files = [
    "weather_airport.csv",
    "weather_central.csv"
]

for filename in weather_files:
    weather = pd.read_csv(DATA_DIR / filename)

    print(f"\n{'=' * 50}")
    print(f"File: {filename}")
    print(f"{'=' * 50}")

    print("Shape:", weather.shape)
    print("\nColumns:", weather.columns.tolist())

    print("\nFirst 5 rows:")
    display(weather.head())

    # Examine temporal coverage
    weather["time"] = pd.to_datetime(weather["time"])

    print("First timestamp:", weather["time"].min())
    print("Last timestamp:", weather["time"].max())
    print("Duplicate timestamps:", weather["time"].duplicated().sum())

    # Compare timestamps with charging observations
    charging_times = set(volume["time"])
    weather_times = set(weather["time"].dropna())

    print("Matching charging timestamps:",
          len(charging_times & weather_times))
    print("Charging timestamps absent from weather:",
          len(charging_times - weather_times))



File: weather_airport.csv
Shape: (4344, 7)

Columns: ['time', 'T', 'P0', 'P', 'U', 'nRAIN', 'Td']

First 5 rows:


,time,T,P0,P,U,nRAIN,Td
0,2022/9/1 0:00,28,755.9,756.2,89,0,26
1,2022/9/1 1:00,29,755.9,756.2,84,0,26
2,2022/9/1 2:00,29,755.1,755.4,79,0,25
3,2022/9/1 3:00,29,755.1,755.4,79,0,25
4,2022/9/1 4:00,28,755.1,755.4,84,0,25


First timestamp: 2022-09-01 00:00:00
Last timestamp: 2023-02-28 23:00:00
Duplicate timestamps: 0
Matching charging timestamps: 4344
Charging timestamps absent from weather: 0

File: weather_central.csv
Shape: (4344, 7)

Columns: ['time', 'T', 'P0', 'P', 'U', 'nRAIN', 'Td']

First 5 rows:


,time,T,P0,P,U,nRAIN,Td
0,2022/9/1 0:00,27.3,750.6,756.1,87,0,24.9
1,2022/9/1 1:00,27.3,750.6,756.1,87,0,24.9
2,2022/9/1 2:00,27.3,750.6,756.1,87,0,24.9
3,2022/9/1 3:00,27.3,750.6,756.1,87,0,24.9
4,2022/9/1 4:00,27.3,750.6,756.1,87,0,24.9


First timestamp: 2022-09-01 00:00:00
Last timestamp: 2023-02-28 23:00:00
Duplicate timestamps: 0
Matching charging timestamps: 4344
Charging timestamps absent from weather: 0


**Weather Station Data Summary**

| Property                    | Airport station     | Central station     |
| --------------------------- | ------------------- | ------------------- |
| Rows                        | 4,344               | 4,344               |
| Columns                     | 7                   | 7                   |
| Missing timestamps          | 0                   | 0                   |
| Duplicate timestamps        | 0                   | 0                   |
| Matches charging timestamps | Yes                 | Yes                 |
| Observation period          | Sep 2022 – Feb 2023 | Sep 2022 – Feb 2023 |

The two datasets contain six weather variables. Their definitions are documented in the repository's `weather_header.txt`.

**Weather Variable Descriptions**

| Variable | Meaning                                     | Unit |
| -------- | ------------------------------------------- | ---- |
| `T`      | Air temperature                             | °C   |
| `P0`     | Atmospheric pressure at the weather station | mmHg |
| `P`      | Atmospheric pressure reduced to sea level   | mmHg |
| `U`      | Relative humidity                           | %    |
| `nRAIN`  | Rain intensity category                     | 0–3  |
| `Td`     | Dew point temperature                       | °C   |

**Note:** the weather measurements come from two locations, not from all 275 zones. 

### Understanding the spatial matrices

We now examine two files that describe relationships between geographic zones:

- `adj.csv`: Which zones are geographically connected or adjacent?
- `distance.csv`: How far apart are the zones?

In [9]:

# Load geographic matrices
adj = pd.read_csv(DATA_DIR / "adj.csv")
distance = pd.read_csv(DATA_DIR / "distance.csv")

for name, df in [
    ("Adjacency matrix", adj),
    ("Distance matrix", distance)
]:
    print(f"\n{'=' * 45}")
    print(name)
    print(f"{'=' * 45}")

    print("Shape:", df.shape)

    print("\nFirst 5 column names:")
    print(df.columns[:5].tolist())

    print("\nFirst 5 rows and columns:")
    display(df.iloc[:5, :5])

    print("Columns match charging zones:",
          set(df.columns) == set(volume.columns) - {"time"})



Adjacency matrix
Shape: (275, 275)

First 5 column names:
['102', '104', '105', '106', '107']

First 5 rows and columns:


,102,104,105,106,107
0,1,0,0,0,0
1,0,1,1,1,0
2,0,1,1,1,1
3,0,1,1,1,1
4,0,0,1,1,1


Columns match charging zones: True

Distance matrix
Shape: (275, 275)

First 5 column names:
['102', '104', '105', '106', '107']

First 5 rows and columns:


,102,104,105,106,107
0,0.000000,1809.441169,2196.926362,1991.889627,2863.407789
1,1809.441169,0.000000,734.061580,1274.183390,1919.472076
2,2196.926362,734.061580,0.000000,782.769630,1211.703811
3,1991.889627,1274.183390,782.769630,0.000000,877.584708
4,2863.407789,1919.472076,1211.703811,877.584708,0.000000


Columns match charging zones: True


**Adjacency and Distance Matrix Summary**

| Property                  | `adj.csv`         | `distance.csv`         |
| ------------------------- | ----------------- | ---------------------- |
| Shape                     | 275 × 275         | 275 × 275              |
| Represents                | Zone connectivity | Distance between zones |
| Values                    | 0 or 1            | Numerical distances    |
| Matches charging zone IDs | Yes               | Yes                    |
| Diagonal values shown     | 1                 | 0                      |


1. Adjacency matrix (`adj.csv`)
This matrix describes which geographic zones are connected.
For example, the first rows indicate that zone 104 is connected to zones 105 and 106, while zone 102 is not connected to 104.
A value of 1 on the diagonal means that a zone is considered connected to itself in this matrix representation.

2. Distance matrix (`distance.csv`)
This matrix describes the distance between pairs of zones.
For example, the output shows:
- Zones 102 and 104: approximately 1,809 distance units.
- Zones 104 and 105: approximately 734 distance units.
- A zone's distance from itself: 0.

### Explore the Points of Interest (POI) data

Now we examine `poi.csv`, the largest file in our dataset.

Points of Interest describe urban locations such as restaurants, residential buildings and lifestyle services.

According to the paper, these were collected in December 2022 using AMap and grouped into three broad categories.

In [10]:
poi = pd.read_csv(DATA_DIR / "poi.csv")

print("Shape:", poi.shape)

print("\nColumns:")
print(poi.columns.tolist())

print("\nFirst 5 rows:")
display(poi.head())

print("\nData types:")
display(poi.dtypes.to_frame("dtype"))


Shape: (712135, 3)

Columns:
['primary_types', 'longitude', 'latitude']

First 5 rows:


,primary_types,longitude,latitude
0,lifestyle services,114.576628,22.494887
1,lifestyle services,114.579614,22.492192
2,lifestyle services,114.565933,22.500528
3,lifestyle services,114.577400,22.492460
4,lifestyle services,114.574730,22.494576



Data types:


,dtype
primary_types,str
longitude,float64
latitude,float64


Each row represents a Point of Interest (POI), with its category and geographic location.

There is no TAZID column. Unlike `inf.csv`, the POIs aren't directly assigned to the 275 charging zones.

To examine how many POIs are located within each zone, we'd eventually need geographic zone boundaries and a spatial join.

There's another detail worth remembering. The repository documents different coordinate systems for station-level locations (GCJ-02) and POIs (WGS84). We'll need to account for that when mapping or combining them.

#### What types of POIs are included?

In [11]:

# Count POIs by category
poi_categories = (
    poi["primary_types"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="count")
)

# Calculate percentage of total
poi_categories["percentage"] = (
    poi_categories["count"] / len(poi) * 100
).round(2)

display(poi_categories)

print("Total POI records:", len(poi))
print("Number of categories:", poi["primary_types"].nunique())

print("\nMissing values:")
display(poi.isna().sum())

print("\nDuplicate rows:", poi.duplicated().sum())


,category,count,percentage
0,lifestyle services,393428,55.25
1,business and residential,183383,25.75
2,food and beverage services,135324,19.00


Total POI records: 712135
Number of categories: 3

Missing values:


primary_types    0
longitude        0
latitude         0
dtype: int64


Duplicate rows: 12


1. Lifestyle services dominate the POI dataset. They account for more than half of all POI records. However, this doesn't necessarily mean they occupy more geographic area.

2. There are no missing values in the POI file. All 712,135 records contain a category, longitude and latitude.

3. Only 12 rows are exact duplicates. That's a tiny fraction of the dataset.

### Dataset catalog

We finish with a compact catalog that summarizes the dimensions of every CSV file, including `inf_raw.csv`, which we haven't inspected individually.

In [12]:

# Categorize all CSV files
file_groups = {
    "volume.csv": "Charging time series",
    "volume-11kW.csv": "Charging time series",
    "duration.csv": "Charging time series",
    "occupancy.csv": "Charging time series",
    "e_price.csv": "Charging time series",
    "s_price.csv": "Charging time series",
    "weather_airport.csv": "Weather",
    "weather_central.csv": "Weather",
    "inf.csv": "Station information",
    "inf_raw.csv": "Station information",
    "poi.csv": "Points of interest",
    "adj.csv": "Spatial matrices",
    "distance.csv": "Spatial matrices",
}

# Build a catalog of files and their dimensions
records = []

for filename, group in file_groups.items():
    path = DATA_DIR / filename
    df = pd.read_csv(path)

    records.append({
        "file": filename,
        "group": group,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "size_mb": round(path.stat().st_size / 1024**2, 2)
    })

catalog = pd.DataFrame(records)

display(catalog)

print("Total files:", len(catalog))
print("Total size (MB):", catalog["size_mb"].sum().round(2))


,file,group,rows,columns,size_mb
0,volume.csv,Charging time series,4344,276,15.74
1,volume-11kW.csv,Charging time series,4344,276,18.81
2,duration.csv,Charging time series,4344,276,16.98
3,occupancy.csv,Charging time series,4344,276,5.26
4,e_price.csv,Charging time series,4344,276,11.49
5,s_price.csv,Charging time series,4344,276,14.27
6,weather_airport.csv,Weather,4344,7,0.16
7,weather_central.csv,Weather,4344,7,0.17
8,inf.csv,Station information,1362,7,0.07
9,inf_raw.csv,Station information,1587,7,0.08


Total files: 13
Total size (MB): 115.17


- `inf.csv` contains 1,362 station records after filtering.

- `inf_raw.csv` contains 1,587 station records associated with the selected 275 zones.

This is different from the 1,682 stations described in the original dataset. The repository's `inf_raw.csv` is not necessarily the complete original raw dataset.